# Question 4: Searching and RAG

Prepare and index the chunks as in Question 3, then search for `python function definition` and identify the book behind the top result.

This notebook runs independently. Run the cells in order using the project's Python environment. `minsearch` and `gitsource` are already project dependencies.

In [ ]:
from pathlib import Path
from gitsource import chunk_documents
from minsearch import Index

# Support running from the repository root or homework/.
project_root = Path.cwd()
if not (project_root / "books_text").is_dir():
    project_root = project_root.parent

books_text_dir = project_root / "books_text"
markdown_files = sorted(books_text_dir.glob("*.md"))
if not markdown_files:
    raise FileNotFoundError(f"No Markdown books found in {books_text_dir}")

## Load and chunk all books

Keep non-empty lines and apply the same 100-line window and 50-line step as Question 2.

In [ ]:
books = []
for markdown_path in markdown_files:
    lines = markdown_path.read_text(encoding="utf-8").splitlines()
    books.append({
        "source": markdown_path.name,
        "content": [line for line in lines if line.strip()],
    })

chunks = chunk_documents(books, size=100, step=50)
print(f"Created {len(chunks):,} chunks from {len(books)} books.")

## Prepare searchable documents

MinSearch expects text strings. Join each chunk’s lines with newlines, preserving its source and starting position without modifying the original chunk.

In [ ]:
def prepare_documents(chunks):
    documents = []
    for chunk in chunks:
        document = chunk.copy()
        document["content"] = "\n".join(chunk["content"])
        documents.append(document)
    return documents


documents = prepare_documents(chunks)

## Build the search index

In [ ]:
index = Index(text_fields=["content"])
index.fit(documents)
print(f"Indexed {len(documents):,} chunks.")

## Search the index

Request the top five results, ordered by relevance.

In [ ]:
results = index.search("python function definition", num_results=5)

for rank, result in enumerate(results, start=1):
    print(f"{rank}. {result['source']} (starting line index: {result['start']})")

## Inspect the top result and answer

The `source` field identifies the Markdown book that supplied the chunk.

In [ ]:
if not results:
    raise ValueError("The search returned no results.")

top_result = results[0]
print(f"Top result source: {top_result['source']}")
print("\nTop result content:\n")
print(top_result["content"])